## MOLECULAR DESCRIPTORS CALCULATION (3D) - Eg with Mordred

#### 1. Set-up:

In [55]:
## packages to install if required (remove hashtag and run):
#!pip install pysmiles
#!pip install descriptastorus
#!pip install mordred


In [56]:
## import necessary libraries:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from pysmiles import read_smiles

from rdkit import Chem
#from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem, PandasTools

from mordred import Calculator, descriptors

from descriptastorus.descriptors.DescriptorGenerator import MakeGenerator


In [57]:
## export dataframe with substances + SMILES:
smiles_df = pd.read_csv('data/contaminants.csv', delimiter = ";")
smiles_df.head(5)


,name,substance_type,CAS_nb,SMILES,detected
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1


In [58]:
## data cleaning:
print(f"Length of database with NAs: {len(smiles_df.SMILES)}")

if smiles_df.SMILES.isna().sum() > 0:
    smiles_cleaned1 = smiles_df.loc[smiles_df.SMILES.notna()].copy()#smiles_df.dropna() ## remove NAs (cannot compute molecular descriptor if no SMILES structure present)
    print(f"Length of database without NAs (cleaned version): {len(smiles_cleaned1.SMILES)}")
else:
    None


Length of database with NAs: 70
Length of database without NAs (cleaned version): 67


In [59]:
## extract list of molecules from their SMILES:
smiles_list = smiles_cleaned1.SMILES

mol_list = []

for s in smiles_list:
    mol = Chem.MolFromSmiles(s)
    mol_list.append(mol)


In [60]:
## append to df + rename columns with mols to 'molecule'
smiles_cleaned = pd.concat([smiles_cleaned1.reset_index(drop = True), pd.DataFrame(mol_list).reset_index(drop = True)], axis = 1)
smiles_cleaned = smiles_cleaned.rename(columns = {0: "molecule"})
smiles_cleaned


,name,substance_type,CAS_nb,SMILES,detected,molecule
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1,<rdkit.Chem.rdchem.Mol object at 0x17d35af10>
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d35b610>
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1,<rdkit.Chem.rdchem.Mol object at 0x17d35a490>
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d359460>
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d359e70>
...,...,...,...,...,...,...
62,tramadol,PPCP,27203-92-5,CN(C)C[C@H]1CCCC[C@@]1(C2=CC(=CC=C2)OC)O,0,<rdkit.Chem.rdchem.Mol object at 0x17d4458c0>
63,chloroacetic acid,solvent,79-11-8,C(C(=O)O)Cl,0,<rdkit.Chem.rdchem.Mol object at 0x17d445930>
64,DMSO,solvent,67-68-5,CS(=O)C,0,<rdkit.Chem.rdchem.Mol object at 0x17d4459a0>
65,methanol,solvent,67-56-1,CO,0,<rdkit.Chem.rdchem.Mol object at 0x17d445a10>


#### 2. Calculation of molecular descriptors:

In [61]:
## set-up Mordred calculator:
calculator_mordred = Calculator(descriptors, ignore_3D = True)


In [62]:
## calculate molecular descriptors:
md_df = calculator_mordred.pandas(mol_list)


 10%|████▌                                       | 7/67 [00:06<01:59,  2.00s/it]

/Users/ciloesans56/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 31%|█████████████▍                             | 21/67 [00:16<00:30,  1.52it/s]

/Users/ciloesans56/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|███████████████████████████████████████████| 67/67 [00:22<00:00,  2.92it/s]

/Users/ciloesans56/anaconda3/lib/python3.11/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


In [63]:
## reframe df (error on last line => bug fixed)
df = pd.concat([smiles_cleaned.reset_index(drop = True), pd.DataFrame(md_df).reset_index(drop = True)], axis = 1)
df


,name,substance_type,CAS_nb,SMILES,detected,molecule,ABC,ABCGG,nAcid,nBase,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1,<rdkit.Chem.rdchem.Mol object at 0x17d35af10>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.496496,46.853672,201.078979,7.733807,362,21,74.0,85.0,4.694444,3.472222
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d35b610>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.677653,50.667067,348.926284,12.031941,612,26,86.0,97.0,8.256944,4.180556
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1,<rdkit.Chem.rdchem.Mol object at 0x17d35a490>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,10.344674,84.588084,415.074199,8.831366,2244,39,146.0,172.0,10.201389,6.166667
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d359460>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.675269,50.806847,317.953661,11.355488,603,26,90.0,103.0,6.666667,4.000000
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d359e70>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,10.165967,54.227403,367.909603,12.686538,744,35,106.0,126.0,8.569444,4.229167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,tramadol,PPCP,27203-92-5,CN(C)C[C@H]1CCCC[C@@]1(C2=CC(=CC=C2)OC)O,0,<rdkit.Chem.rdchem.Mol object at 0x17d4458c0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,...,9.918425,52.447194,263.188529,5.981557,670,30,96.0,112.0,7.006944,4.291667
63,chloroacetic acid,solvent,79-11-8,C(C(=O)O)Cl,0,<rdkit.Chem.rdchem.Mol object at 0x17d445930>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,1,0,...,6.834109,27.254130,93.982157,11.747770,18,2,16.0,14.0,3.361111,1.333333
64,DMSO,solvent,67-68-5,CS(=O)C,0,<rdkit.Chem.rdchem.Mol object at 0x17d4459a0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,6.188264,24.179697,78.013936,7.801394,9,0,12.0,9.0,3.111111,1.000000
65,methanol,solvent,67-56-1,CO,0,<rdkit.Chem.rdchem.Mol object at 0x17d445a10>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,1.098612,7.493061,32.026215,5.337702,1,0,2.0,1.0,2.0,1.000000


#### 3. Cleaning of MD full of NAs (remove these columns):

In [64]:
mordred_md = pd.concat([smiles_cleaned, pd.DataFrame(md_df)], axis = 1)
mordred_md


,name,substance_type,CAS_nb,SMILES,detected,molecule,ABC,ABCGG,nAcid,nBase,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1,<rdkit.Chem.rdchem.Mol object at 0x17d35af10>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.496496,46.853672,201.078979,7.733807,362,21,74.0,85.0,4.694444,3.472222
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d35b610>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.677653,50.667067,348.926284,12.031941,612,26,86.0,97.0,8.256944,4.180556
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1,<rdkit.Chem.rdchem.Mol object at 0x17d35a490>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,10.344674,84.588084,415.074199,8.831366,2244,39,146.0,172.0,10.201389,6.166667
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d359460>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,9.675269,50.806847,317.953661,11.355488,603,26,90.0,103.0,6.666667,4.000000
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d359e70>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,10.165967,54.227403,367.909603,12.686538,744,35,106.0,126.0,8.569444,4.229167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,tramadol,PPCP,27203-92-5,CN(C)C[C@H]1CCCC[C@@]1(C2=CC(=CC=C2)OC)O,0,<rdkit.Chem.rdchem.Mol object at 0x17d4458c0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,...,9.918425,52.447194,263.188529,5.981557,670,30,96.0,112.0,7.006944,4.291667
63,chloroacetic acid,solvent,79-11-8,C(C(=O)O)Cl,0,<rdkit.Chem.rdchem.Mol object at 0x17d445930>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,1,0,...,6.834109,27.254130,93.982157,11.747770,18,2,16.0,14.0,3.361111,1.333333
64,DMSO,solvent,67-68-5,CS(=O)C,0,<rdkit.Chem.rdchem.Mol object at 0x17d4459a0>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,6.188264,24.179697,78.013936,7.801394,9,0,12.0,9.0,3.111111,1.000000
65,methanol,solvent,67-56-1,CO,0,<rdkit.Chem.rdchem.Mol object at 0x17d445a10>,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,...,1.098612,7.493061,32.026215,5.337702,1,0,2.0,1.0,2.0,1.000000


In [65]:
## convert df back to numeric (so descriptors with errors can be converted to NAs & then later removed):
mordred_md_num = mordred_md.iloc[:, 5:-1].apply(pd.to_numeric, errors = "coerce")

mordred_md_clean = mordred_md_num.dropna(axis = 1, how = "all")
mordred_md_clean


,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,VE1_A,VE2_A,...,SRW09,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1
0,0,0,19.544560,2.377815,4.755629,19.544560,1.302971,3.627536,3.457764,0.230518,...,0.000000,9.496496,46.853672,201.078979,7.733807,362,21,74.0,85.0,4.694444
1,0,0,20.971017,2.377461,4.754921,20.971017,1.165057,3.775189,3.759106,0.208839,...,0.000000,9.677653,50.667067,348.926284,12.031941,612,26,86.0,97.0,8.256944
2,0,0,34.992970,2.613976,4.943050,34.992970,1.249749,4.277221,3.188891,0.113889,...,8.444838,10.344674,84.588084,415.074199,8.831366,2244,39,146.0,172.0,10.201389
3,0,0,22.540937,2.377662,4.755325,22.540937,1.252274,3.804096,3.839997,0.213333,...,0.000000,9.675269,50.806847,317.953661,11.355488,603,26,90.0,103.0,6.666667
4,0,0,24.107183,2.519762,5.039524,24.107183,1.205359,3.919405,3.804569,0.190228,...,0.000000,10.165967,54.227403,367.909603,12.686538,744,35,106.0,126.0,8.569444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,0,1,24.017666,2.462551,4.925102,24.017666,1.264088,3.859038,3.738528,0.196765,...,0.000000,9.918425,52.447194,263.188529,5.981557,670,30,96.0,112.0,7.006944
63,1,0,5.226252,1.847759,3.695518,5.226252,1.045250,2.408576,2.130986,0.426197,...,0.000000,6.834109,27.254130,93.982157,11.747770,18,2,16.0,14.0,3.361111
64,0,0,3.464102,1.732051,3.464102,3.464102,0.866025,2.178059,1.931852,0.482963,...,0.000000,6.188264,24.179697,78.013936,7.801394,9,0,12.0,9.0,3.111111
65,0,0,2.000000,1.000000,2.000000,2.000000,1.000000,1.407606,1.414214,0.707107,...,0.000000,1.098612,7.493061,32.026215,5.337702,1,0,2.0,1.0,2.000000


In [66]:
## append MD to original df (now that MD are cleaned => rows full of NAs removed):
mordred_df = pd.concat([smiles_cleaned, mordred_md_clean], axis = 1)
mordred_df


,name,substance_type,CAS_nb,SMILES,detected,molecule,nAcid,nBase,SpAbs_A,SpMax_A,...,SRW09,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1
0,carbaryl,insecticide,63-25-2,CNC(=O)OC1=CC=CC2=CC=CC=C21,1,<rdkit.Chem.rdchem.Mol object at 0x17d35af10>,0,0,19.544560,2.377815,...,0.000000,9.496496,46.853672,201.078979,7.733807,362,21,74.0,85.0,4.694444
1,chlorpyrifos,insecticide,2921-88-2,CCOP(=S)(OCC)OC1=NC(=C(C=C1Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d35b610>,0,0,20.971017,2.377461,...,0.000000,9.677653,50.667067,348.926284,12.031941,612,26,86.0,97.0,8.256944
2,cypermethrine,insecticide,52315-07-8,CC1(C(C1C(=O)OC(C#N)C2=CC(=CC=C2)OC3=CC=CC=C3)...,1,<rdkit.Chem.rdchem.Mol object at 0x17d35a490>,0,0,34.992970,2.613976,...,8.444838,10.344674,84.588084,415.074199,8.831366,2244,39,146.0,172.0,10.201389
3,DDD(2-4),insecticide,72-54-8,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)C(Cl)Cl)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d359460>,0,0,22.540937,2.377662,...,0.000000,9.675269,50.806847,317.953661,11.355488,603,26,90.0,103.0,6.666667
4,dicofol,insecticide,115-32-2,C1=CC(=CC=C1C(C2=CC=C(C=C2)Cl)(C(Cl)(Cl)Cl)O)Cl,1,<rdkit.Chem.rdchem.Mol object at 0x17d359e70>,0,0,24.107183,2.519762,...,0.000000,10.165967,54.227403,367.909603,12.686538,744,35,106.0,126.0,8.569444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,tramadol,PPCP,27203-92-5,CN(C)C[C@H]1CCCC[C@@]1(C2=CC(=CC=C2)OC)O,0,<rdkit.Chem.rdchem.Mol object at 0x17d4458c0>,0,1,24.017666,2.462551,...,0.000000,9.918425,52.447194,263.188529,5.981557,670,30,96.0,112.0,7.006944
63,chloroacetic acid,solvent,79-11-8,C(C(=O)O)Cl,0,<rdkit.Chem.rdchem.Mol object at 0x17d445930>,1,0,5.226252,1.847759,...,0.000000,6.834109,27.254130,93.982157,11.747770,18,2,16.0,14.0,3.361111
64,DMSO,solvent,67-68-5,CS(=O)C,0,<rdkit.Chem.rdchem.Mol object at 0x17d4459a0>,0,0,3.464102,1.732051,...,0.000000,6.188264,24.179697,78.013936,7.801394,9,0,12.0,9.0,3.111111
65,methanol,solvent,67-56-1,CO,0,<rdkit.Chem.rdchem.Mol object at 0x17d445a10>,0,0,2.000000,1.000000,...,0.000000,1.098612,7.493061,32.026215,5.337702,1,0,2.0,1.0,2.000000


#### 4. Export results to .csv file:

In [67]:
mordred_df.to_csv("data/mordred_descriptors.csv", index = False)


#### A1. Simple example of Mordred on 3 molecular structures (from SMILES):

https://github.com/mordred-descriptor/mordred/tree/develop/examples


In [24]:
#### 1. ONE MOLECULE + ONE DESCRIPTOR:
from rdkit import Chem

from mordred import Chi, ABCIndex

benzene = Chem.MolFromSmiles('c1ccccc1')

# create descriptor instance
abci = ABCIndex.ABCIndex()

# calculate descriptor value
result = abci(benzene)

print(str(abci), result)

# create descriptor instance with parameter
chi_pc4 = Chi.Chi(type='path_cluster', order=4)

# calculate
result = chi_pc4(benzene)

print(str(chi_pc4), result)


AttributeError: module 'numpy' has no attribute 'float'.
`np.float` was a deprecated alias for the builtin `float`. To avoid this error in existing code, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

In [25]:
#### 2. ONE MOLECULE + MULTIPLE DESCRIPTORS:
from multiprocessing import freeze_support

from rdkit import Chem

from mordred import Chi, ABCIndex, RingCount, Calculator, is_missing, descriptors

if __name__ == "__main__":
    freeze_support()

    benzene = Chem.MolFromSmiles("c1ccccc1")

    # Create empty Calculator instance
    calc1 = Calculator()

    # Register descriptor instance
    calc1.register(Chi.Chi(type="path_cluster", order=4))

    # Register descriptor class using preset
    calc1.register(RingCount.RingCount)

    # Register all descriptors in module
    calc1.register(ABCIndex)

    # Calculate descriptors
    result = calc1(benzene)

    print(result)
    # >>> [0.0, 1, 0, 0, 0, 1, (snip)

    # Calculator constructor can register descriptors
    calc2 = Calculator(Chi.Chi)

    # Descriptors module contains all descriptors
    calc3 = Calculator(descriptors)

    # User can access all descriptor instances by descriptors property
    print(calc3.descriptors)
    # >>> (mordred.EccentricConnectivityIndex.EccentricConnectivityIndex(), (snip)

    # Calculate descriptors
    result = calc3(benzene)

    # get first missing value
    na1 = next(r for r in result if is_missing(r))

    # get reason
    print(na1.error)
    # >>> missing 3D coordinate

    # Delete all missing value
    result = result.drop_missing()

    # convert to dict
    print(result.asdict())


Result({'Xpc-4d': 0.0, 'nRing': 1, 'n3Ring': 0, 'n4Ring': 0, 'n5Ring': 0, 'n6Ring': 1, 'n7Ring': 0, 'n8Ring': 0, 'n9Ring': 0, 'n10Ring': 0, 'n11Ring': 0, 'n12Ring': 0, 'nG12Ring': 0, 'nHRing': 0, 'n3HRing': 0, 'n4HRing': 0, 'n5HRing': 0, 'n6HRing': 0, 'n7HRing': 0, 'n8HRing': 0, 'n9HRing': 0, 'n10HRing': 0, 'n11HRing': 0, 'n12HRing': 0, 'nG12HRing': 0, 'naRing': 1, 'n3aRing': 0, 'n4aRing': 0, 'n5aRing': 0, 'n6aRing': 1, 'n7aRing': 0, 'n8aRing': 0, 'n9aRing': 0, 'n10aRing': 0, 'n11aRing': 0, 'n12aRing': 0, 'nG12aRing': 0, 'naHRing': 0, 'n3aHRing': 0, 'n4aHRing': 0, 'n5aHRing': 0, 'n6aHRing': 0, 'n7aHRing': 0, 'n8aHRing': 0, 'n9aHRing': 0, 'n10aHRing': 0, 'n11aHRing': 0, 'n12aHRing': 0, 'nG12aHRing': 0, 'nARing': 0, 'n3ARing': 0, 'n4ARing': 0, 'n5ARing': 0, 'n6ARing': 0, 'n7ARing': 0, 'n8ARing': 0, 'n9ARing': 0, 'n10ARing': 0, 'n11ARing': 0, 'n12ARing': 0, 'nG12ARing': 0, 'nAHRing': 0, 'n3AHRing': 0, 'n4AHRing': 0, 'n5AHRing': 0, 'n6AHRing': 0, 'n7AHRing': 0, 'n8AHRing': 0, 'n9AHRing': 0

In [26]:
#### 3. MULTIPLE MOLECULES + MULTIPLE DESCRIPTORS:

from multiprocessing import freeze_support

from rdkit import Chem

from mordred import Calculator, descriptors

if __name__ == "__main__":
    freeze_support()

    mols = [
        Chem.MolFromSmiles("c1ccccc1"),
        Chem.MolFromSmiles("c1ccccc1Cl"),
        Chem.MolFromSmiles("c1ccccc1C"),
    ]

    # Create Calculator
    calc = Calculator(descriptors)

    # map method calculate multiple molecules (return generator)
    print(list(calc.map(mols)))

    # pandas method calculate multiple molecules (return pandas DataFrame)
    print(calc.pandas(mols))


100%|█████████████████████████████████████████████| 3/3 [00:00<00:00,  3.67it/s]

[Result(<rdkit.Chem.rdchem.Mol object at 0x1696eab20>,[<mordred.error.Error object at 0x168e1a3b0>, <mordred.error.Error object at 0x168e1a410>, 0, 0, 8.0, 2.0000000000000004, 4.0, 8.0, 1.3333333333333333, 2.6876239260353, 2.449489742783178, 0.40824829046386296, 0.38505411084803687, 14.696938456699069, 2.4494897427831783, 2.176813580076092, 6, 6, 12, 6, 0, 0, 0, 6, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 54.0, 54.0, 54.0, 27.0, 0.0, 0.0, 0.0, 0.0, 0.0, 30.0, 36.0, 48.0, 42.0, 18.0, 3.0, 0.0, 0.0, 0.0, 30.0, 36.0, 48.0, 42.0, 18.0, 3.0, 0.0, 0.0, 0.0, 222.0, 252.0, 288.0, 186.0, 42.0, 3.0, 0.0, 0.0, 0.0, 871.68111, 938.2272539999998, 1010.8697819999996, 584.1738029999999, 78.738912, 3.0481920000000002, 0.0, 0.0, 0.0, 2727.6038770815603, 3229.52110871909, 3917.9408069422025, 2833.8925682797944, 874.9221648086934, 93.25123329279079, 0.0, 0.0, 0.0, 85.55387999999999, 87.948888, 130.65468, 148.34391599999998, 83.01657599999999, 20.155392, 0.0, 0.0, 0.0, 68.055, 72.675, 106.335, 115.86750000000002,


100%|█████████████████████████████████████████████| 3/3 [00:00<00:00,  3.62it/s]

                                                 ABC  \
0  module 'numpy' has no attribute 'float'.\n`np....   
1  module 'numpy' has no attribute 'float'.\n`np....   
2  module 'numpy' has no attribute 'float'.\n`np....   

                                               ABCGG  nAcid  nBase   SpAbs_A  \
0  module 'numpy' has no attribute 'float'.\n`np....      0      0  8.000000   
1  module 'numpy' has no attribute 'float'.\n`np....      0      0  8.720566   
2  module 'numpy' has no attribute 'float'.\n`np....      0      0  8.720566   

    SpMax_A  SpDiam_A    SpAD_A   SpMAD_A   LogEE_A  ...     SRW10     TSRW10  \
0  2.000000  4.000000  8.000000  1.333333  2.687624  ...  7.627057  30.941317   
1  2.101003  4.202006  8.720566  1.245795  2.844305  ...  8.124151  33.544698   
2  2.101003  4.202006  8.720566  1.245795  2.844305  ...  8.124151  33.544698   

           MW       AMW  WPath  WPol  Zagreb1  Zagreb2  mZagreb1  mZagreb2  
0   78.046950  6.503913     27     3     24.0     24